# SAR signal and scattering computational companion

**Foundation:** [SM-FND-012 — Remote sensing physics and radiometry](../foundations/remote-sensing-physics/radar-scattering-and-sar-signal-formation.md)  
**Level:** introductory to intermediate  
**Dependencies:** NumPy  
**Data:** entirely synthetic; no satellite observations or real geographic measurements are used.

This lab makes selected SAR concepts computationally tangible: range resolution, complex signal representation, coherent scattering, speckle statistics, multilooking, interferometric coherence and phase-to-range sensitivity. It does **not** simulate a complete SAR acquisition or focus raw satellite data.

## 1. Range resolution from waveform bandwidth

For the simplified ideal relation, slant-range resolution is approximately \(c/(2B)\). This is not a delivered product pixel size or a full sensor model.

In [ ]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)
c = 299_792_458.0  # m/s
bandwidth_hz = np.array([10e6, 20e6, 40e6])
slant_range_resolution_m = c / (2.0 * bandwidth_hz)
print('Bandwidth (MHz):', bandwidth_hz / 1e6)
print('Ideal slant-range resolution (m):', slant_range_resolution_m)
assert np.all(np.diff(slant_range_resolution_m) < 0)
assert np.isclose(slant_range_resolution_m[1], c / (40e6))

## 2. Range from two-way travel time

The pulse travels to the target and back, so the simplified range relation contains a factor of two.

In [ ]:
round_trip_time_s = np.array([100e-6, 200e-6, 500e-6])
slant_range_m = c * round_trip_time_s / 2.0
print('Round-trip time (microseconds):', round_trip_time_s * 1e6)
print('Slant range (km):', slant_range_m / 1000.0)
assert np.all(slant_range_m > 0)

## 3. Complex samples: I/Q, amplitude, intensity and phase

A complex sample \(z=I+jQ\) contains both magnitude and phase. Detected intensity is magnitude squared.

In [ ]:
z = np.array([3 + 4j, 6 - 8j, -1 + 0j], dtype=np.complex128)
amplitude = np.abs(z)
intensity = np.abs(z) ** 2
phase_rad = np.angle(z)
print('Complex samples:', z)
print('Amplitude:', amplitude)
print('Intensity:', intensity)
print('Phase (rad):', phase_rad)
assert np.allclose(amplitude, [5, 10, 1])
assert np.allclose(intensity, [25, 100, 1])
assert np.isclose(phase_rad[0], np.arctan2(4, 3))

## 4. Coherent summation and speckle

A resolution cell can contain many scatterers. Their complex returns add as vectors; random relative phase creates variable amplitude and intensity even when the scatterer population is statistically identical from cell to cell. This is an idealised fully developed speckle experiment.

In [ ]:
rng = np.random.default_rng(20261121)
n_cells = 50_000
n_scatterers = 80
phases = rng.uniform(0.0, 2.0 * np.pi, size=(n_cells, n_scatterers))
scatterer_amplitudes = np.ones((n_cells, n_scatterers))
cell_complex_return = np.sum(scatterer_amplitudes * np.exp(1j * phases), axis=1) / np.sqrt(n_scatterers)
single_look_intensity = np.abs(cell_complex_return) ** 2
mean_intensity = single_look_intensity.mean()
intensity_cv = single_look_intensity.std(ddof=1) / mean_intensity
print('Mean intensity:', mean_intensity)
print('Intensity coefficient of variation:', intensity_cv)
print('Ideal fully developed single-look CV: approximately 1')
assert 0.94 < intensity_cv < 1.06

## 5. Multilooking reduces variance at a resolution cost

For independent, identically distributed exponential intensity looks, averaging \(L\) looks gives an idealised coefficient of variation of \(1/\sqrt{L}\). This experiment shows the statistical trade-off; real SAR looks may be correlated and processing-dependent.

In [ ]:
n_trials = 200_000
look_counts = np.array([1, 2, 4, 8])
cv_empirical = []
for looks in look_counts:
    # Unit-mean exponential intensity is an ideal single-look model.
    averaged = rng.exponential(scale=1.0, size=(n_trials, looks)).mean(axis=1)
    cv_empirical.append(averaged.std(ddof=1) / averaged.mean())
cv_empirical = np.asarray(cv_empirical)
cv_theoretical = 1.0 / np.sqrt(look_counts)
print('Looks:', look_counts)
print('Empirical CV:', cv_empirical)
print('Ideal CV:', cv_theoretical)
assert np.allclose(cv_empirical, cv_theoretical, rtol=0.035)

## 6. Interferometric coherence and phase

A simple complex correlation experiment constructs two observations with a controlled shared component, phase offset and independent component. The estimated normalised cross-correlation approximates the shared coherence magnitude; its phase reflects the imposed offset under this synthetic convention.

In [ ]:
n = 100_000
rho = 0.85
imposed_phase_rad = 0.7
z1 = (rng.normal(size=n) + 1j * rng.normal(size=n)) / np.sqrt(2.0)
independent = (rng.normal(size=n) + 1j * rng.normal(size=n)) / np.sqrt(2.0)
z2 = np.exp(1j * imposed_phase_rad) * (rho * z1 + np.sqrt(1.0 - rho**2) * independent)
cross = np.sum(z1 * np.conjugate(z2))
coherence_estimate = np.abs(cross) / np.sqrt(np.sum(np.abs(z1)**2) * np.sum(np.abs(z2)**2))
phase_estimate_rad = np.angle(cross)
print('Target shared-component coefficient:', rho)
print('Estimated coherence magnitude:', coherence_estimate)
print('Imposed phase (rad):', imposed_phase_rad)
print('Estimated cross-product phase (rad):', phase_estimate_rad)
assert abs(coherence_estimate - rho) < 0.015
assert abs(abs(phase_estimate_rad) - imposed_phase_rad) < 0.015

## 7. Phase sensitivity to line-of-sight path change

For the simplified two-way path relation, the phase-change magnitude is \(4\pi\Delta R/\lambda\). The sign depends on the processor's phase and displacement convention. This calculation is not a displacement retrieval from real data.

In [ ]:
wavelength_m = 0.056
path_change_m = np.array([0.005, 0.010, 0.014, 0.028])
phase_change_rad = 4.0 * np.pi * path_change_m / wavelength_m
print('Path change (mm):', path_change_m * 1000)
print('Phase-change magnitude (rad):', phase_change_rad)
print('One full 2π phase cycle corresponds to path change λ/2 (mm):', wavelength_m * 500)
assert np.isclose(phase_change_rad[2], np.pi, atol=1e-12)

## 8. Slant-range to ground-range resolution

For a flat reference surface and incidence angle \(\theta\) from the surface normal, the simplified ground-range resolution is \(\delta_{gr}=\delta_r/\sin\theta\). This is not valid as a full terrain model on slopes.

In [ ]:
slant_resolution_m = 5.0
incidence_deg = np.array([20.0, 35.0, 45.0, 60.0])
ground_resolution_m = slant_resolution_m / np.sin(np.deg2rad(incidence_deg))
print('Incidence angle (degrees):', incidence_deg)
print('Approximate ground-range resolution (m):', ground_resolution_m)
assert np.all(ground_resolution_m >= slant_resolution_m)
assert np.all(np.diff(ground_resolution_m) < 0)

## 9. Interpretation and limits

- These experiments use synthetic values and idealised assumptions; they do not emulate a mission's waveform, antenna, orbit, focusing processor, calibration or terrain correction.
- The exponential intensity model assumes fully developed single-look speckle over a homogeneous target. Heterogeneity, correlated looks, thermal noise and processing alter the distribution.
- The coherence experiment uses an ideal controlled complex-correlation model. Real interferometric coherence is estimated over spatial windows and affected by temporal, geometric and volume decorrelation, noise and processing.
- A phase-to-path calculation does not separate deformation from topography, atmosphere, orbit error, scattering changes or unwrapping errors.
- Resolution formulas are simplified theoretical relationships; pixel spacing and delivered resolution are not interchangeable.

**Next step:** use the [SAR acquisition method](../collection/sar-acquisition-and-observation-design.md) to select and document a real product only after the foundation and acquisition records have been reviewed. The Nairobi scene specification remains unexecuted.